# Backward refinement

**Table of contents**<a id='toc0_'></a>    
- 1. [Load](#toc1_)    
- 2. [Settings](#toc2_)    
- 3. [Solve](#toc3_)    
- 4. [Save](#toc4_)    

<!-- vscode-jupyter-toc-config
	numbering=true
	anchor=true
	flat=false
	minLevel=2
	maxLevel=6
	/vscode-jupyter-toc-config -->
<!-- THIS CELL WILL BE REPLACED ON TOC UPDATE. DO NOT WRITE YOUR TEXT IN THIS CELL -->

In [1]:
import numpy as np
import torch

In [2]:
from EconDLSolvers import choose_gpu
from LifeCycleModel import LifeCycleModelClass

## 1. <a id='toc1_'></a>[Load](#toc0_)

In [3]:
folder = '../output/'
model_simult = LifeCycleModelClass(device='cpu',load=folder+'LargeLifeCycleModel.pt')

## 2. <a id='toc2_'></a>[Settings](#toc0_)

In [4]:
device, _ = choose_gpu()

GPU 0: 67.91GB free [NVIDIA H100 80GB HBM3]
Best GPU: 0


In [5]:
train = {}
train['backward'] = True
train['use_simult_in_backward'] = True
train['NN_init_std'] = 0.001
train['Nneurons_value_t'] = np.array([50,50])
train['Nneurons_policy_t'] = np.array([50,50])
train['N'] = 1_000_000
train['N_target_batches'] = 1_000
train['batch_size'] = 100_000
train['K_time'] = 4 * 60.0

## 3. <a id='toc3_'></a>[Solve](#toc0_)

In [ ]:
model_backward = LifeCycleModelClass(device=device,algoname='DeepVPDDCBackward',train=train)
model_backward.solve(model_simult=model_simult,do_print=True)

started solving: 2026-02-11 11:15:28
t =  19
 training policy network 
  epoch =     0: 5.06580e+00 
  epoch =   100: 5.06630e+00 
  epoch =   200: 5.06628e+00 
  epoch =   300: 5.06576e+00 
  epoch =   400: 5.06600e+00 
  epoch =   500: 5.06624e+00 
  epoch =   600: 5.06630e+00 
  epoch =   700: 5.06638e+00 
  epoch =   800: 5.06608e+00 
  epoch =   900: 5.06576e+00 
  epoch =   999: 5.06625e+00 
t =  18
 training value network 
  epoch =     0: 5.8e-05
  epoch =     9: 5.8e-05 time limit reached [365.0 secs]
 training policy network 
  epoch =     0: 4.17149e+00 
  epoch =   100: 4.14844e+00 
  epoch =   122:     4.1 time limit reached [361.8 secs]
t =  17
 training value network 
  epoch =     0: 4.8e-05
  epoch =     0: 4.8e-05 time limit reached [917.8 secs]
 training policy network 
  epoch =     0: 5.20146e+00 
  epoch =   100: 5.18106e+00 
  epoch =   122:     5.2 time limit reached [362.4 secs]
t =  16
 training value network 
  epoch =     0: 8.7e-05
  epoch =     0: 8.7e-05 

In [ ]:
model_backward.simulate_R()
model_backward.more_simulation_outcomes()
print(model_backward.sim.R)

In [ ]:
model_backward.compute_euler_errors(Nbatch_share=0.0001)

## 4. <a id='toc4_'></a>[Save](#toc0_)

In [ ]:
vars = ['reward','actions','DC','outcomes','taste_shocks','shocks','states_pd','states']
for var in vars: setattr(model_backward.train,var,None)
model_backward.save(folder + 'LargeLifeCycleModel_backward.pt')